# 06 · Federation (no-ETL) — DuckDB vs Trino lendo a CLEAN do cliente → grafo

**Path FEDERATION:** se a CLEAN é **do cliente** e ele não quer ETL, dá pra uma engine
(DuckDB **ou** Trino) **ler direto** o lake dele (Delta **ou** DuckLake) e alimentar o
FalkorDB, **sem copiar nada** pra nossa RAW/CLEAN? Alimenta a pergunta *duckdb vs trino* de
[pontos-a-verificar §2](../../docs/arquitetura/2.0-lake-aberto/pontos-a-verificar.md).

| Engine | Formato do cliente | Lê direto? |
|---|---|---|
| **DuckDB** | Delta | ✅ `delta_scan(...)` |
| **DuckDB** | DuckLake | ✅ `ATTACH 'ducklake:…' (READ_ONLY)` |
| **Trino** | Delta | ✅ connector `delta_lake` + `register_table` |
| **Trino** | DuckLake | 🛑 **sem connector** (formato é específico do DuckDB) |

> Scripts: [`federate_duckdb.py`](federate_duckdb.py), [`federate_trino.py`](federate_trino.py).
> Resultados e caveats em [RESULTADOS.md](RESULTADOS.md).
> ⚠️ Requer FalkorDB em `localhost:6380`; o script Trino sobe um container Trino.

In [ ]:
import sys; sys.path.insert(0, "/Users/allanfraga/Repos/strattum/experimentacoes")
import exp

## DuckDB — lê os dois formatos in-process (a engine mais flexível)
Mesmo motor que o `memory_worker` já usa → encaixa direto (só troca o `_build_select` do
reader pra apontar no lake do cliente). Lê Delta **e** DuckLake, sem infra extra.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "federate_duckdb.py"], cwd="06-federation-read-engine")
# DuckDB lê Delta e DuckLake do cliente -> 50 linhas -> 50 nós no FalkorDB ✅

## Trino — quando o cliente já tem Trino / lake distribuído
Lê Delta (e Iceberg, Snowflake… via connectors), mas exige catálogo/metastore e **não lê
DuckLake**. Cliente em DuckLake ⇒ engine de federação **tem que ser DuckDB**.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "federate_trino.py"], cwd="06-federation-read-engine")
# Trino lê Delta do cliente -> grafo ✅ ; DuckLake -> ❌ sem connector

## Conclusão
✅ Path federação validado no essencial. A **engine é ditada pelo formato do cliente**:
DuckLake ⇒ DuckDB obrigatório; Delta/Iceberg/Snowflake ⇒ DuckDB **ou** Trino. Falta cravar
contra **lake real em S3** e resolver **dedup/freshness** da releitura (federação relê a
fonte toda vez).